## Unsupervised Anomaly Detection

**Objective:** Implement unsupervised models without referencing the target label (is_fraud) to detect novel, unlabelled anomaly patterns (*Zero-Day Fraud*).

<details>
<summary><b>📖 Methodological Justification: Raw Dataset Reuse (Click to expand)</b></summary>

Relying exclusively on supervised models limits detection to known fraud types. This stage processes raw data directly (fraudTrain.csv and fraudTest.csv) rather than reusing supervised features because:

1. **Objective Incompatibility:** Supervised preprocessing optimizes features for tree-based splits against labels. Unsupervised models (Isolation Forest, LOF) depend on spatial geometry, distances, and point density.
2. **Anomaly-Centric Features:** Unsupervised algorithms perform better with features emphasizing extreme behavior (Frequency Encoding, cyclic temporal dynamics, metric deviations).
3. **Spatial Structure:** Preserves the original anomaly signature without smoothing outlier signals.
</details>

In [3]:
# Import the required libraries and utility functions
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

from src.utils import calculate_age, calculate_distance_km, transform_cyclic_hour

In [4]:
# Load the datasets
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

# Combine both datasets into a single DataFrame
df_raw = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# Separate the target variable
y_true = df_raw['is_fraud']
X_raw = df_raw.drop(columns=['is_fraud'])

print("X_raw dimensions:", X_raw.shape)
print("y_true dimensions:", y_true.shape)

X_raw dimensions: (1852394, 22)
y_true dimensions: (1852394,)


In [5]:
# Since we only want legitimate transactions and both y_true and X_raw share the same indexes:
# Create a mask using the target variable
mask_legit = (y_true == 0)

# Apply the mask to X_raw while preserving index alignment
X_legit_raw = X_raw[mask_legit].copy()

# Verify that no records were lost during filtering
total_rows = len(X_raw)
legitimate_rows = len(X_legit_raw)
fraud_count = (y_true == 1).sum()

print(f"Original total rows: {total_rows:,}")
print(f"Rows in X_legit_raw:      {legitimate_rows:,}")
print(f"Number of fraud cases:  {fraud_count:,}")
print(f"Legitimate rows + fraud cases total: {legitimate_rows + fraud_count:,}")

Original total rows: 1,852,394
Rows in X_legit_raw:      1,842,743
Number of fraud cases:  9,651
Legitimate rows + fraud cases total: 1,852,394


In [6]:
# Inspect the available columns
X_legit_raw.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long'],
      dtype='str')

## Feature Engineering & Transformation Criteria

* **Dropped Features:** Unnamed: 0, trans_num, cc_num, first, last, street.

* **Applied Transformations:**

1. **Transaction Amount (`amt`):**
   Applied `log(1+x)` transformation to reduce extreme right-skewness and improve anomaly detection.

2. **Cyclic Time Features (`trans_date_trans_time`):**
   Applied trigonometric encoding to transaction hours:

   hour_sin = sin(2π × hour / 24)

   hour_cos = cos(2π × hour / 24)

   This preserves the continuity between 23:59 and 00:01.

3. **Geolocation:**
   Calculated Haversine distance (km) between customer and merchant coordinates.

4. **High-Cardinality Categorical Features (`category`, `job`, `state`, `merchant`):**
   Applied Frequency Encoding to identify rare patterns in sparse categories.

5. **Binary Encoding:**
   Converted gender into a binary representation:
   Female = 0, Male = 1.


In [7]:
# Convert transaction timestamp into datetime format
X_legit_raw['trans_date_trans_time'] = pd.to_datetime(X_legit_raw['trans_date_trans_time'])

X_legit_raw['trans_date_trans_time'].dtype

dtype('<M8[us]')

In [8]:
# Extract transaction hour
hours = X_legit_raw['trans_date_trans_time'].dt.hour

# Transform transaction hour into cyclic features
(
    X_legit_raw["hour_sin"],
    X_legit_raw["hour_cos"],
) = transform_cyclic_hour(X_legit_raw)

# Verify that sine and cosine values remain between -1 and 1
X_legit_raw[['hour_sin', 'hour_cos']].describe()

,hour_sin,hour_cos
count,1.842743e+06,1.842743e+06
mean,-1.371902e-01,-1.845884e-02
std,6.939641e-01,7.065780e-01
min,-1.000000e+00,-1.000000e+00
25%,-8.660254e-01,-7.071068e-01
50%,-2.588190e-01,-1.836970e-16
75%,5.000000e-01,7.071068e-01
max,1.000000e+00,1.000000e+00


In [9]:
# Calculate the distance between customer and merchant locations
X_legit_raw["distance_km"] = calculate_distance_km(X_legit_raw)

In [10]:
# Calculate customer age at transaction time
X_legit_raw["age"] = calculate_age(X_legit_raw)

In [11]:
# Convert gender into binary values: Female = 0, Male = 1
X_legit_raw['gender'] = X_legit_raw['gender'].map({'F': 0, 'M': 1})

In [12]:
# Apply Frequency Encoding using global category frequencies
cols_freq = ["category", "job", "state", "merchant"]

for col in cols_freq:
    # Normalize frequencies to values between 0 and 1
    freq_map = X_legit_raw[col].value_counts(normalize=True)

    # Create the encoded feature
    X_legit_raw[f"{col}_freq"] = X_legit_raw[col].map(freq_map)

# Verify the newly generated frequency-encoded features
X_legit_raw[[f"{col}_freq" for col in cols_freq]].head()

,category_freq,job_freq,state_freq,merchant_freq
0,0.048554,0.002770,0.023302,0.000946
1,0.094404,0.003942,0.014605,0.001896
2,0.072623,0.000395,0.004342,0.001431
3,0.101619,0.001976,0.009083,0.002017
4,0.061814,0.001579,0.022512,0.001239


In [13]:
# Inspect the current DataFrame columns after feature engineering
X_legit_raw.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'hour_sin', 'hour_cos', 'distance_km', 'age',
       'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [14]:
# Drop unnecessary columns
cols_to_drop = [
    # Encoded features
    "category",
    "job",
    "state",
    "merchant",
    # Processed features
    "trans_date_trans_time",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "dob",
    # Irrelevant features
    "street",
    "city",
    "zip",
    "unix_time",
    "first",
    "last",
    "cc_num",
    "trans_num",
    "Unnamed: 0",
]

X_legit = X_legit_raw.drop(columns=cols_to_drop, errors="ignore")

X_legit.columns

Index(['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km',
       'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [15]:
# Apply logarithmic scaling to transaction amount to reduce skewness
# Tree-based models do not require scaling, but this transformation improves interpretability
X_legit["amt"] = np.log1p(X_legit["amt"])

### Isolation Forest Justification
* **Extreme Imbalance:** Fraud represents only $\sim 0.52\%$ of total volume.
* **Geometric Isolation:** Explicitly isolates anomalies using random partitioning without requiring prior labels.
* **Scalability:** Computational efficiency to process $\sim 1.8\text{M}$ records smoothly.

In [16]:
# Train Isolation Forest only on legitimate transactions
iso_forest = IsolationForest(
    n_estimators=300, contamination=0.01, random_state=42, n_jobs=-1
)

iso_forest.fit(X_legit)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [17]:
# Map predictions:
preds_legit = iso_forest.predict(X_legit)

# -1 becomes 1 (Anomaly)
# 1 becomes 0 (Normal transaction)
preds_legit_binary = [1 if p == -1 else 0 for p in preds_legit]

In [18]:
# Ideally, the model should not detect any anomalies because all transactions in this subset are legitimate
# and should therefore be considered "normal". However, Isolation Forest does not have access to the true labels,
# so it will still flag statistically unusual observations as anomalies.

print(sum(preds_legit_binary))

# Calculate the anomaly percentage within legitimate transactions
print((sum(preds_legit_binary) / len(X_legit)) * 100)

18428
1.000030932148433


In [19]:
# Prepare the fraud subset for evaluation
mask_fraud = y_true == 1
X_fraud_raw = X_raw[mask_fraud].copy()

total_rows = len(X_raw)
fraud_rows = len(X_fraud_raw)
legitimate_count = (y_true == 0).sum()

print(f"Original total rows: {total_rows:,}")
print(f"Rows in X_fraud_raw:      {fraud_rows:,}")
print(f"Number of legitimate transactions:  {legitimate_count:,}")
print(f"Fraud rows + legitimate transactions: {fraud_rows + legitimate_count:,}")

Original total rows: 1,852,394
Rows in X_fraud_raw:      9,651
Number of legitimate transactions:  1,842,743
Fraud rows + legitimate transactions: 1,852,394


In [20]:
# Apply the same preprocessing pipeline used previously
X_fraud_raw['trans_date_trans_time'] = pd.to_datetime(X_fraud_raw['trans_date_trans_time'])

# Extract the transaction hour
hours = X_fraud_raw['trans_date_trans_time'].dt.hour

(
    X_fraud_raw["hour_sin"],
    X_fraud_raw["hour_cos"],
) = transform_cyclic_hour(X_fraud_raw)

# Generate the 'age' and 'distance_km' features and encode 'gender'
X_fraud_raw["distance_km"] = calculate_distance_km(X_fraud_raw)
X_fraud_raw["age"] = calculate_age(X_fraud_raw)
X_fraud_raw['gender'] = X_fraud_raw['gender'].map({'F': 0, 'M': 1})

for col in cols_freq:
# Normalize frequencies to values between 0 and 1
    freq_map = X_fraud_raw[col].value_counts(normalize=True)

# Create the encoded feature
    X_fraud_raw[f"{col}_freq"] = X_fraud_raw[col].map(freq_map)

# Apply log1p to reduce skewness and compress extreme transaction amounts
X_fraud_raw["amt"] = np.log1p(X_fraud_raw["amt"])

# Inspect the resulting columns
X_fraud_raw.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'hour_sin', 'hour_cos', 'distance_km', 'age',
       'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [21]:
# Drop unnecessary columns
X_fraud = X_fraud_raw.drop(columns=cols_to_drop, errors="ignore")

X_fraud.columns

Index(['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km',
       'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [22]:
# Predict on the real fraud transactions
preds_fraud = iso_forest.predict(X_fraud)

# Map predictions:
# -1 → 1 (Anomaly/Fraud)
#  1 → 0 (Normal)
preds_fraud_binary = [1 if p == -1 else 0 for p in preds_fraud]

print(f"Fraud cases detected after training only on legitimate transactions: {sum(preds_fraud_binary)}")

Fraud cases detected after training only on legitimate transactions: 547


> **Observation:** Training Isolation Forest exclusively on legitimate transactions resulted in poor fraud detection performance. The next experiment retrains the model using the complete dataset (legitimate + fraudulent transactions).

In [23]:
# Apply the same preprocessing pipeline used previously
X_raw['trans_date_trans_time'] = pd.to_datetime(X_raw['trans_date_trans_time'])

# Extract the transaction hour
hours = X_raw['trans_date_trans_time'].dt.hour

# Transform the transaction hour into cyclic features
(
    X_raw["hour_sin"],
    X_raw["hour_cos"],
) = transform_cyclic_hour(X_raw)

# Generate the 'age' and 'distance_km' features and encode 'gender'
X_raw["distance_km"] = calculate_distance_km(X_raw)
X_raw["age"] = calculate_age(X_raw)
X_raw['gender'] = X_raw['gender'].map({'F': 0, 'M': 1})

for col in cols_freq:
# Normalize frequencies to values between 0 and 1
    freq_map = X_raw[col].value_counts(normalize=True)

# Create the encoded feature
    X_raw[f"{col}_freq"] = X_raw[col].map(freq_map)

# Apply log1p to reduce skewness and compress extreme transaction amounts
X_raw["amt"] = np.log1p(X_raw["amt"])

X_raw.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'hour_sin', 'hour_cos', 'distance_km', 'age',
       'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [24]:
# Drop unnecessary columns
X_raw = X_raw.drop(columns=cols_to_drop, errors="ignore")

X_raw.columns

Index(['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km',
       'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [25]:
# Train Isolation Forest on the full dataset
iso_forest_total = IsolationForest(
    n_estimators=300, contamination=0.01, random_state=42, n_jobs=-1
)

iso_forest_total.fit(X_raw)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [26]:
# Predict on the real fraud transactions
preds_legit_total = iso_forest_total.predict(X_raw)

# Map predictions:
# -1 → 1 (Anomaly/Fraud)
#  1 → 0 (Normal)
preds_legit_total_binary = [1 if p == -1 else 0 for p in preds_legit_total]

print(sum(preds_legit_total_binary))

18524


> **Evaluation Question:** Although the model now flags 18,524 anomalies, the key question is how many correspond to true frauds and how many fraudulent transactions remain undetected.

In [27]:
# Evaluate the final fraud detection rate
captured_frauds = sum(
    1 for pred, real in zip(preds_legit_total_binary, y_true) if pred == 1 and real == 1
)

total_frauds = (y_true == 1).sum()

print(f"Total fraud cases: {total_frauds:,}")
print(f"Frauds detected by the model: {captured_frauds:,}")
print(f"Fraud detection rate (Recall): {(captured_frauds / total_frauds) * 100:.2f}%")

Total fraud cases: 9,651
Frauds detected by the model: 216
Fraud detection rate (Recall): 2.24%


> **Experiment Note:** Training Isolation Forest strictly on legitimate transactions yielded poor recall. Retraining across the full dataset (legitimate + fraud).

> **Evaluation:** Out of the 18,524 flagged records, how many are true frauds vs. missed cases?

> **Conclusion:** Isolation Forest is unsuited for this specific feature space topology due to high false-positive overlap.